<a href="https://colab.research.google.com/github/EgzonnOsmanaj/MesoAI/blob/main/KosovoTaxRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install anthropic numpy scikit-learn rank_bm25 tiktoken

In [2]:
import os, sys, json, re, time, hashlib, math
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
import numpy as np
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
import google.generativeai as genai
from google.colab import userdata

# Configure Gemini API
GoogleAIAPI=userdata.get('GoogleAIAPI') # Assumes key is set in Colab secrets
genai.configure(api_key=GoogleAIAPI)

# Initialize Gemini models
gemini_haiku_model = genai.GenerativeModel('gemini-2.5-flash') # Equivalent to Anthropic's Haiku
gemini_sonnet_model = genai.GenerativeModel('gemini-flash-latest') # Equivalent to Anthropic's Sonnet

# Print available models for debugging
print("Available Gemini models:")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(f"  - {m.name}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Available Gemini models:
  - models/gemini-2.5-flash
  - models/gemini-2.5-pro
  - models/gemini-2.0-flash
  - models/gemini-2.0-flash-001
  - models/gemini-2.0-flash-lite-001
  - models/gemini-2.0-flash-lite
  - models/gemini-2.5-flash-preview-tts
  - models/gemini-2.5-pro-preview-tts
  - models/gemma-4-26b-a4b-it
  - models/gemma-4-31b-it
  - models/gemini-flash-latest
  - models/gemini-flash-lite-latest
  - models/gemini-pro-latest
  - models/gemini-2.5-flash-lite
  - models/gemini-2.5-flash-image
  - models/gemini-3-pro-preview
  - models/gemini-3-flash-preview
  - models/gemini-3.1-pro-preview
  - models/gemini-3.1-pro-preview-customtools
  - models/gemini-3.1-flash-lite-preview
  - models/gemini-3.1-flash-lite
  - models/gemini-3-pro-image-preview
  - models/gemini-3-pro-image
  - models/nano-banana-pro-preview
  - models/gemini-3.1-flash-image-preview
  - models/gemini-3.1-flash-image
  - models/gemini-3.5-flash
  - models/lyria-3-clip-preview
  - models/lyria-3-pro-preview
  - 

In [3]:
# ==========================================
# 1. MOCK KNOWLEDGE BASE (Kosovo Tax Corpus)
# ==========================================
KOSOVO_TAX_CORPUS = [
    {
        "document_id": "LAW_CIT_06_L105",
        "title": "Law No. 06/L-105 on Corporate Income Tax",
        "sections": [
            {
                "parent_id": "CIT_ART_7",
                "header": "Article 7 - Tax Rates",
                "text": "Corporate income tax shall be charged at the rate of ten percent (10%) on taxable income. For taxpayers with gross annual income up to fifty thousand Euros (50,000 EUR), small business tax statements apply under specific regimes.",
                "children": [
                    "Corporate income tax is charged at the rate of ten percent (10%) on taxable income.",
                    "Taxpayers with gross annual income up to fifty thousand Euros (50,000 EUR) are subject to small business tax regimes."
                ]
            },
            {
                "parent_id": "CIT_ART_14",
                "header": "Article 14 - Non-Deductible Expenses",
                "text": "For the purposes of determining taxable income, no deduction shall be allowed for: 1. Interest paid that exceeds the limit specified under transfer pricing rules; 2. Fines, penalties, and administrative punitive damages issued by ATK or public authorities; 3. Representation expenses exceeding 1% of total gross income.",
                "children": [
                    "No deduction is allowed for interest exceeding limits specified under transfer pricing rules.",
                    "Fines, penalties, and administrative punitive damages issued by ATK or public authorities are non-deductible.",
                    "Representation expenses exceeding 1% of total gross income are strictly non-deductible."
                ]
            }
        ]
    },
    {
        "document_id": "LAW_VAT_05_L037",
        "title": "Law No. 05/L-037 on Value Added Tax",
        "sections": [
            {
                "parent_id": "VAT_ART_26",
                "header": "Article 26 - VAT Rates",
                "text": "The standard VAT rate for taxable supplies of goods and services, as well as imports in Kosovo, is eighteen percent (18%). A reduced VAT rate of eight percent (8%) applies to specific supplies including water, electricity, basic foodstuffs, and textbooks.",
                "children": [
                    "The standard VAT rate for taxable supplies of goods, services, and imports in Kosovo is eighteen percent (18%).",
                    "A reduced VAT rate of eight percent (8%) applies to supplies including water, electricity, basic foodstuffs, and textbooks."
                ]
            }
        ]
    }
]


In [4]:
# ── 2. Data Structures ────────────────────────────────────────────────────────
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    title: str
    source: str
    category: str
    text: str
    embedding: Optional[List[float]] = field(default=None, repr=False)

@dataclass
class RetrievalResult:
    chunk: Chunk
    bm25_score: float = 0.0
    vector_score: float = 0.0
    hybrid_score: float = 0.0
    rerank_score: float = 0.0

In [5]:
# ── 3. Chunking ───────────────────────────────────────────────────────────────
def semantic_chunk(doc: Dict, max_chars: int = 800, overlap_chars: int = 150) -> List[Chunk]:
    """
    Semantic / paragraph-boundary chunking with overlap.
    Why this approach:
      - Legal documents are paragraph-structured; splitting mid-paragraph loses
        the logical unit (a rule, rate, or condition).
      - 150-char overlap preserves cross-paragraph context so a chunk about
        "the penalty rate" also contains the phrase "late payment interest"
        from the preceding paragraph.
      - 800-char ceiling keeps chunks within ~150 tokens, well within the
        context window budget while still dense enough for TF-IDF to work.
    """
    text = doc["text"].strip()
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
    chunks, current, idx = [], "", 0
    for para in paragraphs:
        if current and len(current) + len(para) + 2 > max_chars:
            chunk_text = current.strip()
            chunks.append(Chunk(
                chunk_id=f"{doc['id']}_c{idx}", doc_id=doc["id"],
                title=doc["title"], source=doc["source"],
                category=doc["category"], text=chunk_text,
            ))
            idx += 1
            tail = chunk_text[-overlap_chars:] if len(chunk_text) > overlap_chars else chunk_text
            current = tail + "\n\n" + para
        else:
            current = (current + "\n\n" + para).lstrip() if current else para
    if current.strip():
        chunks.append(Chunk(
            chunk_id=f"{doc['id']}_c{idx}", doc_id=doc["id"],
            title=doc["title"], source=doc["source"],
            category=doc["category"], text=current.strip(),
        ))
    return chunks

def build_chunks(corpus_docs: List[Dict]) -> List[Chunk]:
    all_chunks = []
    for doc_entry in corpus_docs:
        doc_id = doc_entry["document_id"]
        doc_title = doc_entry["title"]
        doc_category = "Tax Law" # Default category, or infer if available in data

        for section in doc_entry["sections"]:
            # Create a simplified 'doc' dictionary for semantic_chunk
            section_doc = {
                "id": f"{doc_id}_{section['parent_id']}", # Unique ID for this section as a 'document'
                "title": f"{doc_title} - {section['header']}", # More specific title
                "source": doc_id, # Reference back to the original document
                "category": doc_category,
                "text": section["text"],
            }
            all_chunks.extend(semantic_chunk(section_doc))

    avg_chars = sum(len(c.text) for c in all_chunks) // len(all_chunks) if all_chunks else 0
    print(f"[Chunking] {len(corpus_docs)} docs \u2192 {len(all_chunks)} chunks  "
          f"(avg {avg_chars} chars)")
    return all_chunks

In [6]:
# 4. Embeddings (TF-IDF + Cosine)
# Design note: In production, replace with voyage-law-2 or text-embedding-3-large
# for dense semantic search. TF-IDF is highly competitive on legal text
# (explicit legal terminology matters more than semantic paraphrase) and
# removes the need for an embedding API, making the pipeline fully offline.
_TFIDF_VEC: Optional[TfidfVectorizer] = None
_EMBED_CACHE: Dict[str, List[float]] = {}

def fit_vectorizer(corpus_texts: List[str]) -> TfidfVectorizer:
    global _TFIDF_VEC
    if _TFIDF_VEC is None:
        _TFIDF_VEC = TfidfVectorizer(
            max_features=512, ngram_range=(1, 2),
            sublinear_tf=True, strip_accents='unicode', min_df=1
        )
        _TFIDF_VEC.fit(corpus_texts)
        print(f"[TF-IDF] Fitted on {len(corpus_texts)} texts; vocab={len(_TFIDF_VEC.vocabulary_)}")
    return _TFIDF_VEC

def embed(texts: List[str]) -> List[List[float]]:
    """Embed texts; uses cache to avoid recomputing."""
    global _EMBED_CACHE
    new_texts = [t for t in texts if hashlib.md5(t.encode()).hexdigest() not in _EMBED_CACHE]
    if new_texts:
        vec = _TFIDF_VEC or fit_vectorizer(texts)
        vecs = vec.transform(new_texts).toarray().tolist()
        for t, v in zip(new_texts, vecs):
            _EMBED_CACHE[hashlib.md5(t.encode()).hexdigest()] = v
    return [_EMBED_CACHE[hashlib.md5(t.encode()).hexdigest()] for t in texts]

def cosine(a, b) -> float:
    a, b = np.array(a, np.float32), np.array(b, np.float32)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if na and nb else 0.0

# 5. BM25 Index
STOPWORDS = {'the','a','an','is','are','was','were','be','been','have','has','had',
             'do','does','did','will','would','could','should','may','might','must',
             'can','to','of','in','for','on','with','at','by','from','as','or',
             'and','but','not','this','that','these','those','it','its','which'}

def tokenize(text: str) -> List[str]:
    return [t for t in re.findall(r'\b[a-zA-Z0-9]+\b', text.lower())
            if t not in STOPWORDS and len(t) > 1]

class BM25Index:
    """
    BM25 (Best Match 25) index with k1=1.5, b=0.75.
    k1 controls term frequency saturation; b controls field length normalisation.
    These are standard Okapi BM25 defaults and work well for legal text.
    """
    def __init__(self, chunks: List[Chunk]):
        self.chunks = chunks
        self.bm25 = BM25Okapi([tokenize(c.text) for c in chunks], k1=1.5, b=0.75)
        print(f"[BM25] Index built for {len(chunks)} chunks")

    def search(self, query: str, top_k: int = 10) -> List[Tuple[Chunk, float]]:
        scores = self.bm25.get_scores(tokenize(query))
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]
        return [(self.chunks[i], float(s)) for i, s in ranked if s > 0]

In [7]:
# ── 6. Vector Store ───────────────────────────────────────────────────────────
class VectorStore:
    def __init__(self, chunks: List[Chunk]):
        self.chunks = chunks
        fit_vectorizer([c.text for c in chunks])   # fit on corpus first
        embs = embed([c.text for c in chunks])
        for c, e in zip(chunks, embs):
            c.embedding = e
        print(f"[VectorStore] {len(chunks)} chunks embedded (dim={len(embs[0])})")

    def search(self, q_emb: List[float], top_k: int) -> List[Tuple[Chunk, float]]:
        scores = [(c, cosine(q_emb, c.embedding)) for c in self.chunks]
        return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

# ── 7. Hybrid Retriever ───────────────────────────────────────────────────────
class HybridRetriever:
    """
    Combines BM25 and TF-IDF vector scores via min-max normalised linear fusion.
    alpha=0.55 slightly favours the vector signal; legal text benefits from
    both exact-term matching (BM25) and concept-level similarity (vector).
    """
    def __init__(self, chunks: List[Chunk], alpha: float = 0.55):
        self.vs = VectorStore(chunks)
        self.bm = BM25Index(chunks)
        self.chunks = chunks
        self.alpha = alpha

    def retrieve(self, query: str, top_k: int = 8) -> List[RetrievalResult]:
        q_emb = embed([query])[0]
        v_res = {c.chunk_id: s for c, s in self.vs.search(q_emb, top_k * 2)}
        b_res = {c.chunk_id: s for c, s in self.bm.search(query, top_k * 2)}

        def norm(d):
            if not d: return {}
            lo, hi = min(d.values()), max(d.values())
            return d if hi == lo else {k: (v-lo)/(hi-lo) for k, v in d.items()}

        vn, bn = norm(v_res), norm(b_res)
        cmap = {c.chunk_id: c for c in self.chunks}
        combined = {
            cid: RetrievalResult(
                chunk=cmap[cid],
                bm25_score=b_res.get(cid, 0.0),
                vector_score=v_res.get(cid, 0.0),
                hybrid_score=self.alpha * vn.get(cid, 0.0) + (1-self.alpha) * bn.get(cid, 0.0),
            )
            for cid in set(vn) | set(bn)
        }
        return sorted(combined.values(), key=lambda r: r.hybrid_score, reverse=True)[:top_k]


In [8]:
# 8. HyDE Query Rewriting
def hyde_rewrite(question: str) -> str:
    """
    Hypothetical Document Embedding (HyDE):
    Generate a short hypothetical expert answer to the question, then use
    THAT text — not the bare question — as the retrieval query.

    Rationale: Questions ("What is the CIT rate?") have very different word
    distributions than answers ("The CIT rate under Law 05/L-029 is 10%...").
    The hypothetical answer's word distribution closely matches the corpus,
    dramatically improving both BM25 and vector retrieval for short questions.
    """
    resp = gemini_haiku_model.generate_content(
        contents=[
            {"role": "user", "parts": [f'You are a Kosovo tax law expert. Write a SHORT (3-5 sentence) expert ' \
            f'answer to this question using precise legal terminology, specific rates, ' \
            f'and law names. Do NOT say you do not know — write the most plausible answer.\n\n' \
            f'Question: {question}\n\nHypothetical answer:']}
        ],
        generation_config=genai.types.GenerationConfig(max_output_tokens=200)
    )
    return resp.text.strip()

In [9]:
# 9. LLM Reranker
def llm_rerank(question: str, results: List[RetrievalResult],
               top_k: int = 5) -> List[RetrievalResult]:
    """
    Ask a lightweight LLM to score each retrieved chunk 0–10 for relevance
    to the original question (not the HyDE query).

    Using the original question ensures the reranker judges true relevance,
    not similarity to the hypothetical answer. Haiku is used for cost/speed.
    """
    if not results:
        return results
    candidates = "\n\n".join(
        f"[{i+1}] {r.chunk.title}\n{r.chunk.text[:400]}..."
        for i, r in enumerate(results)
    )
    resp = gemini_haiku_model.generate_content(
        contents=[
            {"role": "user", "parts": [f"Score each chunk 0-10 for relevance to this query.\n"
            f"Query: {question}\n\n{candidates}\n\n"
            f"Respond ONLY with a JSON array, e.g. [8,3,9]. No explanation."]}
        ],
        generation_config=genai.types.GenerationConfig(max_output_tokens=100)
    )
    try:
        # Gemini might return markdown, so extract JSON from code block if present
        text_content = resp.text.strip()
        if text_content.startswith('```json') and text_content.endswith('```'):
            text_content = text_content[7:-3].strip()

        m = re.search(r'\[[\d\s,.]+\]', text_content)
        if m:
            scores = json.loads(m.group())
            for i, r in enumerate(results):
                if i < len(scores):
                    r.rerank_score = float(scores[i])
            return sorted(results, key=lambda r: r.rerank_score, reverse=True)[:top_k]
    except Exception as e:
        print(f"Error parsing reranker response: {e}")
        print(f"Raw response: {resp.text}")
    return results[:top_k]

In [10]:
# 10. Grounded Generation
SYSTEM_PROMPT = """You are an expert Kosovo tax advisor grounded in Kosovo's primary legislation.

Rules you must follow:
1. GROUNDED: Only use information from the provided context. Do not invent rates,
   thresholds, dates, or rules not present in the context.
2. PRECISE: Quote specific rates, thresholds, and article/law references.
3. STRUCTURED: Use numbered lists or sections for multi-part answers.
4. HONEST: If the context does not cover the question, say explicitly:
   "This is not covered in the available context. Consult ATK at www.atk-ks.org"
5. CITED: End each key claim with the source law in parentheses."""

def generate(question: str, chunks: List[Chunk], pipeline: str = "enhanced") -> Dict:
    context = "\n\n---\n\n".join(
        f"[Source: {c.title} | {c.source}]\n{c.text}" for c in chunks
    )
    resp = gemini_sonnet_model.generate_content(
        contents=[
            {"role": "user", "parts": [SYSTEM_PROMPT + "\n\n" + f"CONTEXT:\n{context}\n\n---\n\n" \
            f"QUESTION: {question}\n\n" \
            f"Answer strictly from the context above. Cite laws and article numbers."]}
        ],
        generation_config=genai.types.GenerationConfig(max_output_tokens=1000)
    )

    input_tokens = resp.usage_metadata.prompt_token_count if resp.usage_metadata else 0
    output_tokens = resp.usage_metadata.candidates_token_count if resp.usage_metadata else 0

    return {
        "pipeline": pipeline,
        "answer": resp.text,
        "sources": [{"title": c.title, "source": c.source, "chunk_id": c.chunk_id}
                    for c in chunks],
        "tokens_in": input_tokens,
        "tokens_out": output_tokens,
    }

In [11]:
# ── 11. Baseline Pipeline ─────────────────────────────────────────────────────
class BaselineRAG:
    """BM25 keyword search (top-3) + vanilla prompt. No rewriting or reranking."""
    def __init__(self, chunks: List[Chunk]):
        self.bm25 = BM25Index(chunks)
        print("[Baseline] Ready")

    def query(self, question: str, top_k: int = 3) -> Dict:
        t0 = time.time()
        chunks = [r[0] for r in self.bm25.search(question, top_k)]
        result = generate(question, chunks, "baseline")
        result["latency_s"] = round(time.time() - t0, 2)
        result["retrieved_k"] = len(chunks)
        return result

# ── 12. Enhanced Pipeline ─────────────────────────────────────────────────────
class EnhancedRAG:
    """
    Full pipeline: HyDE rewrite → Hybrid retrieval (top-8) →
    LLM reranking (top-5) → Grounded generation with Sonnet.
    """
    def __init__(self, chunks: List[Chunk], alpha: float = 0.55):
        self.retriever = HybridRetriever(chunks, alpha=alpha)
        print("[Enhanced] Ready")

    def query(self, question: str, top_k_ret: int = 8, top_k_rank: int = 5,
              use_hyde: bool = True) -> Dict:
        t0 = time.time()
        rewritten = hyde_rewrite(question) if use_hyde else question
        candidates = self.retriever.retrieve(rewritten, top_k=top_k_ret)
        reranked = llm_rerank(question, candidates, top_k=top_k_rank)
        chunks = [r.chunk for r in reranked]
        result = generate(question, chunks, "enhanced")
        result["latency_s"] = round(time.time() - t0, 2)
        result["retrieved_k"] = len(chunks)
        result["hyde_query"] = rewritten if use_hyde else None
        result["retrieval_scores"] = [
            {"chunk_id": r.chunk.chunk_id, "title": r.chunk.title[:50],
             "bm25": round(r.bm25_score, 4), "vector": round(r.vector_score, 4),
             "hybrid": round(r.hybrid_score, 4), "rerank": round(r.rerank_score, 1)}
            for r in reranked
        ]
        return result


In [12]:
# 13. Evaluation
TEST_QUERIES = [
    {"id": "q01", "type": "simple_fact",      "difficulty": "easy",
     "query": "What is the standard corporate income tax rate in Kosovo?",
     "expected_contains": ["10%"], "relevant_docs": ["cit_001"]},
    {"id": "q02", "type": "simple_fact",      "difficulty": "easy",
     "query": "What are the personal income tax brackets and rates in Kosovo?",
     "expected_contains": ["4%","8%","10%","960"], "relevant_docs": ["pit_001"]},
    {"id": "q03", "type": "procedural",       "difficulty": "easy",
     "query": "When is the deadline to file the annual corporate income tax return?",
     "expected_contains": ["31 March","March"], "relevant_docs": ["admin_001"]},
    {"id": "q04", "type": "multi_doc",        "difficulty": "medium",
     "query": "How are computers and IT equipment depreciated for CIT purposes?",
     "expected_contains": ["25%","Category 2","reducing balance"], "relevant_docs": ["cit_003"]},
    {"id": "q05", "type": "computational",    "difficulty": "medium",
     "query": "A Kosovo resident earns EUR 9,000 annual salary. How much PIT do they owe?",
     "expected_contains": ["633","EUR"], "relevant_docs": ["pit_001"]},
    {"id": "q07", "type": "deep_context",     "difficulty": "hard",
     "query": "What TP documentation must a company maintain if it has EUR 400,000 in related-party transactions?",
     "expected_contains": ["300,000","benchmarking","functional analysis"], "relevant_docs": ["cit_005"]},
    {"id": "q08", "type": "recent_regulation","difficulty": "medium",
     "query": "What changed in Kosovo tax law regarding electronic filing in 2023?",
     "expected_contains": ["EDI","electronic","paper"], "relevant_docs": ["admin_003"]},
    {"id": "q09", "type": "ambiguous",        "difficulty": "medium",
     "query": "Is rental income taxed in Kosovo?",
     "expected_contains": ["9%","600"], "relevant_docs": ["pit_003"]},
    {"id": "q10", "type": "edge_case",        "difficulty": "hard",
     "query": "If I sell my home in Kosovo where I have lived for 3 years, do I pay capital gains tax?",
     "expected_contains": ["exempt","2 years","primary residence"], "relevant_docs": ["pit_004"]},
    {"id": "q11", "type": "cross_border",     "difficulty": "hard",
     "query": "What withholding tax applies when a Kosovo company pays royalties to a UK company?",
     "expected_contains": ["10%","5%","UK","treaty"], "relevant_docs": ["wht_001"]},
    {"id": "q12", "type": "incentives",       "difficulty": "medium",
     "query": "What tax benefits are available for companies in Kosovo Special Economic Zones?",
     "expected_contains": ["5%","Special Economic Zone","customs"], "relevant_docs": ["incentives_001","cit_004"]},
    {"id": "q13", "type": "multi_doc",        "difficulty": "medium",
     "query": "What are total pension and health insurance contribution rates for employees in Kosovo in 2024?",
     "expected_contains": ["5%","3.5%","pension","health"], "relevant_docs": ["pit_002","social_001"]},
    {"id": "q14", "type": "out_of_scope",     "difficulty": "hard",
     "query": "What are the Kosovo inheritance tax rates for property received from a parent?",
     "expected_contains": ["not covered","ATK"], "relevant_docs": []},
    {"id": "q15", "type": "multi_doc",        "difficulty": "hard",
     "query": "How is VAT calculated on imported goods from the EU into Kosovo?",
     "expected_contains": ["18%","customs value","customs duty"], "relevant_docs": ["customs_001","vat_001"]},
]

def retrieval_precision(retrieved_ids: List[str], relevant_docs: List[str]) -> float:
    if not retrieved_ids: return 0.0
    hits = sum(1 for cid in retrieved_ids
               if any(cid.startswith(rdoc) for rdoc in relevant_docs))
    return hits / len(retrieved_ids)

def retrieval_recall(retrieved_ids: List[str], relevant_docs: List[str]) -> float:
    if not relevant_docs: return 1.0
    docs_found = {rdoc for rdoc in relevant_docs
                  for cid in retrieved_ids if cid.startswith(rdoc)}
    return len(docs_found) / len(relevant_docs)

def llm_judge(question: str, answer: str, expected: List[str]) -> Dict:
    """LLM-as-judge: scores answer on correctness, grounding, completeness, clarity."""
    expected_str = ", ".join(f'"{e}"' for e in expected)
    resp = gemini_haiku_model.generate_content(
        contents=[
            {"role": "user", "parts": [f"Evaluate this Kosovo tax law answer. Score each 1-5. Respond ONLY with JSON.\n\n"
            f"Question: {question}\nExpected key terms: {expected_str}\n\n"
            f"Answer:\n{answer[:1200]}\n\n"
            f'Return exactly: {{"correctness":X,"grounding":X,"completeness":X,"clarity":X,' \
            f'"contains_expected":true/false,"comment":"brief"}}']}
        ],
        generation_config=genai.types.GenerationConfig(max_output_tokens=200)
    )
    try:
        text_content = resp.text.strip()
        if text_content.startswith('```json') and text_content.endswith('```'):
            text_content = text_content[7:-3].strip()

        m = re.search(r'\{[^{}]+\}', text_content, re.DOTALL)
        if m: return json.loads(m.group())
    except Exception as e:
        print(f"Error parsing LLM judge response: {e}")
        print(f"Raw response: {resp.text}")
    kw_hits = sum(1 for e in expected if e.lower() in answer.lower())
    s = min(5, max(1, round(kw_hits / max(len(expected), 1) * 5)))
    return {"correctness": s, "grounding": s, "completeness": s, "clarity": 3,
            "contains_expected": kw_hits > 0, "comment": "keyword fallback"}

def run_evaluation(baseline: BaselineRAG, enhanced: EnhancedRAG,
                   queries: List[Dict] = None) -> Dict:
    queries = queries or TEST_QUERIES
    print(f"\n{'='*65}\nEVALUATION — {len(queries)} queries\n{'='*65}\n")
    rows_b, rows_e = [], []
    for t in queries:
        print(f"  [{t['id']}] {t['query'][:60]}...")
        br = baseline.query(t["query"])
        er = enhanced.query(t["query"])
        b_ids = [s["chunk_id"] for s in br["sources"]]
        e_ids = [s["chunk_id"] for s in er["sources"]]
        bp = retrieval_precision(b_ids, t["relevant_docs"])
        br_ = retrieval_recall(b_ids, t["relevant_docs"])
        ep = retrieval_precision(e_ids, t["relevant_docs"])
        er_ = retrieval_recall(e_ids, t["relevant_docs"])
        bj = llm_judge(t["query"], br["answer"], t["expected_contains"])
        ej = llm_judge(t["query"], er["answer"], t["expected_contains"])
        rows_b.append({"id":t["id"],"type":t["type"],"difficulty":t["difficulty"],
                       "ret_p":bp,"ret_r":br_,"ret_f1":2*bp*br_/(bp+br_) if bp+br_ else 0,
                       **{f"gen_{k}":v for k,v in bj.items()},
                       "latency":br["latency_s"],"answer":br["answer"]})
        rows_e.append({"id":t["id"],"type":t["type"],"difficulty":t["difficulty"],
                       "ret_p":ep,"ret_r":er_,"ret_f1":2*ep*er_/(ep+er_) if ep+er_ else 0,
                       **{f"gen_{k}":v for k,v in ej.items()},
                       "latency":er["latency_s"],"answer":er["answer"],
                       "hyde_query":er.get("hyde_query")})
        print(f"    BL  P={bp:.2f} R={br_:.2f} | Corr={bj.get('correctness')}/5  {br['latency_s']}s")
        print(f"    ENH P={ep:.2f} R={er_:.2f} | Corr={ej.get('correctness')}/5  {er['latency_s']}s")
        time.sleep(0.3)

    def avg(rows, key):
        vals = [r[key] for r in rows if isinstance(r.get(key), (int, float))]
        return round(sum(vals)/len(vals), 3) if vals else 0

    summary = {}
    for name, rows in [("baseline", rows_b), ("enhanced", rows_e)]:
        summary[name] = {
            "avg_precision":   avg(rows, "ret_p"),
            "avg_recall":      avg(rows, "ret_r"),
            "avg_f1":          avg(rows, "ret_f1"),
            "avg_correctness": avg(rows, "gen_correctness"),
            "avg_grounding":   avg(rows, "gen_grounding"),
            "avg_completeness":avg(rows, "gen_completeness"),
            "avg_clarity":     avg(rows, "gen_clarity"),
            "avg_latency_s":   avg(rows, "latency"),
        }

    print(f"\n{'─'*65}")
    for name, s in summary.items():
        print(f"\n{name.upper()}")
        print(f"  Retrieval  P={s['avg_precision']} R={s['avg_recall']} F1={s['avg_f1']}")
        print(f"  Generation Correctness={s['avg_correctness']} "
              f"Grounding={s['avg_grounding']} Completeness={s['avg_completeness']}")
        print(f"  Avg latency {s['avg_latency_s']}s")
    return {"summary": summary, "baseline_rows": rows_b, "enhanced_rows": rows_e}

In [15]:
# ── 14. Demo runner ───────────────────────────────────────────────────────────
DEMO_QUESTIONS = [
    "What is the corporate income tax rate in Kosovo, and are there any reduced rates?"
    # "A Kosovo resident earns EUR 9,000 annual salary. Calculate their PIT step by step.",
    # "What withholding tax applies when a Kosovo company pays royalties to a UK company?",
    # "What changed in Kosovo tax law regarding electronic filing and e-invoicing in 2023?",
    # "If I sell my home in Kosovo where I have lived for 3 years, do I pay capital gains tax?",
]

def run_demo(enhanced: EnhancedRAG, baseline: BaselineRAG,
             questions: List[str] = None) -> List[Dict]:
    questions = questions or DEMO_QUESTIONS
    log = []
    print(f"\n{'='*65}\nDEMO LOG\n{'='*65}")
    for q in questions:
        print(f"\nQ: {q}")
        print("—"*60)
        b = baseline.query(q)
        e = enhanced.query(q)
        print(f"[BASELINE]  ({b['latency_s']}s)\n{b['answer']}")
        print(f"\n[ENHANCED]  ({e['latency_s']}s)\n{e['answer']}")
        log.append(
            {
                "question": q,
                "baseline_answer": b["answer"],
                "enhanced_answer": e["answer"],
                "baseline_sources": [s["title"] for s in b["sources"]],
                "enhanced_sources": [s["title"] for s in e["sources"]],
                "hyde_query": e.get("hyde_query", "")[:300],
                "b_latency": b["latency_s"],
                "e_latency": e["latency_s"],
            }
        )
        time.sleep(0.3)
    return log


In [16]:
# 15. Entry point

if __name__ == "__main__":
    print("Kosovo Tax RAG System — Full Pipeline\n")

    # Use the pre-defined KOSOVO_TAX_CORPUS instead of loading from file
    docs = KOSOVO_TAX_CORPUS
    print(f"Loaded {len(docs)} documents from KOSOVO_TAX_CORPUS variable")

    chunks = build_chunks(docs);
    baseline = BaselineRAG(chunks);
    enhanced = EnhancedRAG(chunks)

    # Demo
    demo_log = run_demo(enhanced, baseline)
    with open("demo_log_output.json", "w") as f:
        json.dump(demo_log, f, indent=2, ensure_ascii=False)
    print("\nDemo log → demo_log_output.json")

    # Evaluation (set to False to skip — each call costs API tokens)
    # RUN_EVAL = os.environ.get("RUN_EVAL", "1") == "1"
    RUN_EVAL = False # Explicitly set to False to avoid quota issues
    if RUN_EVAL:
        eval_results = run_evaluation(baseline, enhanced)
        with open("evaluation_results.json", "w") as f:
            json.dump(eval_results, f, indent=2, default=str)
        print("Evaluation results → evaluation_results.json")

Kosovo Tax RAG System — Full Pipeline

Loaded 2 documents from KOSOVO_TAX_CORPUS variable
[Chunking] 2 docs → 3 chunks  (avg 267 chars)
[BM25] Index built for 3 chunks
[Baseline] Ready
[VectorStore] 3 chunks embedded (dim=199)
[BM25] Index built for 3 chunks
[Enhanced] Ready

DEMO LOG

Q: What is the corporate income tax rate in Kosovo, and are there any reduced rates?
————————————————————————————————————————————————————————————
[BASELINE]  (6.09s)
Based on the provided primary legislation, here are the details regarding the corporate income tax rate and any reduced rates in Kosovo:

1. **Standard Corporate Income Tax Rate**: Corporate income tax is charged at

[ENHANCED]  (13.76s)
Based on the provided context, the corporate income tax rate and applicable regimes in Kosovo are as follows:

1. **Standard Corporate Income Tax Rate**: Corporate income tax is charged at a rate of ten

Demo log → demo_log_output.json
